In [0]:
dbutils.widgets.removeAll()

In [0]:
import json
import time
import requests
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
dbutils.widgets.text("storageName", "adlsmartdataproy26")
dbutils.widgets.text("fecha_inicio", "2026-07-01")
dbutils.widgets.text("fecha_fin", "2026-10-31")


In [0]:
catalogo = dbutils.widgets.get("catalogo")
storageName = dbutils.widgets.get("storageName")
fecha_inicio = dbutils.widgets.get("fecha_inicio")
fecha_fin   = dbutils.widgets.get("fecha_fin")


In [0]:

ruta_clima   = f"abfss://raw@{storageName}.dfs.core.windows.net/datasets/clima"
tabla_destino = f"{catalogo}.bronze.clima_raw"

In [0]:
URL_API = "https://archive-api.open-meteo.com/v1/archive"
LATITUD, LONGITUD = 19.4326, -99.1332          # centro de la CDMX
ZONA_HORARIA = "America/Mexico_City"           # las horas del clima quedan en hora local, igual que los viajes
VARIABLES = ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m", "weather_code"]
 
DESCARGAR = True
 
print("Periodo:", fecha_inicio, "->", fecha_fin)
print("Destino raw:", ruta_clima)

Periodo: 2026-07-01 -> 2026-10-31
Destino raw: abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima


In [0]:
def rangos_por_anio(fecha_inicio, fecha_fin):
    """Parte el periodo en un rango por año: [(2026, '2026-07-01', '2026-10-31'), ...]"""
    rangos = []
    for anio in range(int(fecha_inicio[:4]), int(fecha_fin[:4]) + 1):
        inicio = max(fecha_inicio, f"{anio}-07-01")
        fin    = min(fecha_fin,    f"{anio}-10-31")
        rangos.append((anio, inicio, fin))
    return rangos

In [0]:
def descargar(inicio, fin, intentos=4):
    """Llama a la API. Si responde 429 (demasiadas llamadas) espera y reintenta."""
    params = {
        "latitude": LATITUD, "longitude": LONGITUD,
        "start_date": inicio, "end_date": fin,
        "hourly": ",".join(VARIABLES),
        "timezone": ZONA_HORARIA,
    }
    for intento in range(1, intentos + 1):
        r = requests.get(URL_API, params=params, timeout=120)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            time.sleep(10 * intento)
            continue
        raise RuntimeError(f"La API respondió {r.status_code}: {r.text[:200]}")
    raise RuntimeError("La API siguió respondiendo 429 (límite de llamadas). Intenta más tarde.")

In [0]:
if DESCARGAR:
    for anio, inicio, fin in rangos_por_anio(fecha_inicio, fecha_fin):
        datos = descargar(inicio, fin)
        ruta_anio = f"{ruta_clima}/clima_{anio}"
 
        (
            spark.createDataFrame([(json.dumps(datos),)], "value string")
            .coalesce(1)
            .write.mode("overwrite")
            .text(ruta_anio)
        )
 
        print(f"{anio}: {inicio} -> {fin} | {len(datos['hourly']['time'])} horas guardadas en {ruta_anio}")
        time.sleep(1)   # pausa breve entre llamadas
else:
    print("DESCARGAR = False: se usarán los archivos que ya estén en la carpeta clima")

2026: 2026-07-01 -> 2026-10-31 | 2952 horas guardadas en abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026


In [0]:
contenido = [f.name for f in dbutils.fs.ls(ruta_clima) if not f.name.startswith("_")]
print(contenido)
assert len(contenido) >= 1, "No hay datos de clima en la carpeta raw/datasets/clima."

['clima_2026/']


In [0]:
raw = (
    spark.read
    .option("multiLine", "true")
    .option("recursiveFileLookup", "true")
    .json(ruta_clima)
)

In [0]:
bronce = (
    raw
    .withColumn("_archivo_origen", F.col("_metadata.file_path"))
    .withColumn("_fecha_ingesta", F.current_timestamp())
)

In [0]:
# Es una descarga completa del periodo: se reemplaza la tabla en cada ejecución
(
    bronce.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(tabla_destino)
)

In [0]:
b = spark.table(tabla_destino)
b.printSchema()

root
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- hourly: struct (nullable = true)
 |    |-- precipitation: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- relative_humidity_2m: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- temperature_2m: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- weather_code: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- wind_speed_10m: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- hourly_units: struct (nullable = true)
 |    |-- precipitation: string (nullable = true)
 |    |-- relative_humidity_2m: string (nullable = true)
 |    |-- temperature_2m: string (nullable = true)
 |    |-- time: string (nullable = true)
 |    |-- weather

In [0]:
# Una fila por archivo (año) y cuántas horas trae cada una
display(
    b.select(
        "_archivo_origen",
        F.size("hourly.time").alias("horas"),
        F.col("hourly.time")[0].alias("primera_hora"),
        F.element_at("hourly.time", -1).alias("ultima_hora"),
        "hourly_units",
    ).orderBy("_archivo_origen")
)

_archivo_origen,horas,primera_hora,ultima_hora,hourly_units
abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2952,2026-07-01T00:00,2026-10-31T23:00,"List(mm, %, °C, iso8601, wmo code, km/h)"
